# Parsing Strategy

Financial documents come in very different shapes — and different formats. A 10-K annual report is dense prose with scattered tables. An earnings slide deck is one slide per page with almost no prose. SEC EDGAR delivers the same filings as both PDFs and HTML (`.htm`) files.

If we chunk everything the same way, we lose signal — slide text gets split mid-bullet, table rows get merged with prose paragraphs, and retrieval quality suffers.

This notebook implements a **classify-then-parse** pipeline that handles both formats:

```
File (.pdf or .htm)
  └─► parse()          — dispatch on file extension
        ├─ parse_pdf()
        │    ├─► diagnose()   — structural metadata (pages, chars/page, table count)
        │    └─► classify()   — doc type + chunking mode (per_page | overlapping)
        └─ parse_htm()
             └─► classify_htm() — doc type from filename (no scan needed)

Both converge on the same shared helpers:
  ├─ table_to_chunk()      — serialize any 2-D table to markdown
  ├─ prose_to_chunks()     — overlapping windowed prose chunks
  └─ page_to_chunk()       — one chunk per slide page (PDF only)
```

All output is a flat `list[Chunk]` using the shared `Chunk` dataclass from `utils.py`, ready to be embedded and stored in ChromaDB.

## 1. Setup

In [13]:
import sys
import pdfplumber
import re
from pathlib import Path
from bs4 import BeautifulSoup

# utils.py lives in the project root, one level above notebooks/.
# sys.path.insert lets us import it without making the project a package.
sys.path.insert(0, str(Path("..").resolve()))
from utils import Chunk, make_chunk_id

# ── Chunking constants ────────────────────────────────────────────────────────

CHUNK_SIZE = 800        # ~200 tokens — fits most embedding model limits
OVERLAP = 80            # chars repeated at the start of the next chunk
MIN_CHARS = 40          # discard near-empty extractions (headers, footers)
CLASSIFY_PAGES = 3      # pages to scan when keyword-classifying a PDF
SLIDE_CHARS_THRESHOLD = 1500  # below this chars/page → treat as slide deck

PDF_DIR = Path("../datasets/client")

print("Setup complete.")

Setup complete.


## 2. Diagnose

Before we parse anything, we inspect the PDF to understand its structure. This gives us the raw numbers that the classifier uses to make decisions.

| Metric | Why it matters |
|---|---|
| `chars_per_page` | Low → slide deck; high → dense prose report |
| `tables` | High relative to pages → table-heavy document |
| `likely_scan` | True → text extraction won't work, need OCR |
| `size_kb` | Very large → likely has embedded images (e.g. NVIDIA annual report) |

In [14]:
def diagnose(path: Path) -> dict:
    """
    Opens the PDF and collects structural statistics without reading all content.
    Returns a dict of metrics used by classify() to decide how to chunk.
    """
    total_chars = 0
    total_tables = 0
    page_count = 0

    with pdfplumber.open(path) as pdf:
        page_count = len(pdf.pages)

        for page in pdf.pages:
            text = page.extract_text() or ""
            total_chars += len(text)
            total_tables += len(page.find_tables())

    chars_per_page = total_chars / max(page_count, 1)
    likely_scan = chars_per_page < 100

    return {
        "file": path.name,
        "size_kb": round(path.stat().st_size / 1024, 2),
        "pages": page_count,
        "chars_per_page": round(chars_per_page, 1),
        "tables": total_tables,
        "likely_scan": likely_scan,
    }


diagnostics = []
for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    diag = diagnose(pdf_path)
    diagnostics.append(diag)
    print(diag)

{'file': "AMD Q4'25 Earnings Slides FINAL.pdf", 'size_kb': 1214.93, 'pages': 34, 'chars_per_page': 859.0, 'tables': 65, 'likely_scan': False}
{'file': 'Amazon-2025-Annual-Report.pdf', 'size_kb': 1595.95, 'pages': 92, 'chars_per_page': 3457.2, 'tables': 90, 'likely_scan': False}
{'file': 'Meta-12-31-2024-10K-ARS.pdf', 'size_kb': 1847.8, 'pages': 134, 'chars_per_page': 3772.4, 'tables': 59, 'likely_scan': False}
{'file': 'Meta-12-31-2025-Exhibit-99-1-FINAL.pdf', 'size_kb': 172.81, 'pages': 9, 'chars_per_page': 2214.2, 'tables': 8, 'likely_scan': False}
{'file': 'NVIDIA-2025-Annual-Report.pdf', 'size_kb': 48799.4, 'pages': 181, 'chars_per_page': 3675.1, 'tables': 117, 'likely_scan': False}
{'file': 'TSLA-Q1-2026-Update.pdf', 'size_kb': 9767.79, 'pages': 31, 'chars_per_page': 1330.8, 'tables': 18, 'likely_scan': False}
{'file': 'goog-10-q-q1-2025.pdf', 'size_kb': 544.67, 'pages': 49, 'chars_per_page': 2760.2, 'tables': 112, 'likely_scan': False}


## 3. Classify

**PDFs** need structural and text signals to identify their type:
- `chars_per_page < 1500` → slide deck (no text needed)
- Keyword scan of first pages: `"form 10-k"`, `"form 10-q"`, `"exhibit 99"`

**HTM files** don't need any of that — the EDGAR filename already encodes the form type (`AMZN_10-K_2025-02-07.htm`). `classify_htm` just reads the second `_`-delimited segment.

Both return the same `(document_type, chunking_mode)` tuple so the rest of the pipeline is identical.

In [15]:
def classify(path: Path, diag: dict) -> tuple[str, str]:
    """Classify a PDF by structure + keyword scan of its first few pages."""

    if diag["chars_per_page"] < SLIDE_CHARS_THRESHOLD:
        return "earnings_slides", "per_page"

    sample_text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages[:CLASSIFY_PAGES]:
            sample_text += (page.extract_text() or "").lower()

    if "form 10-k" in sample_text or "annual report" in sample_text:
        return "10-K", "overlapping"
    if "form 10-q" in sample_text:
        return "10-Q", "overlapping"
    if "exhibit 99" in sample_text:
        return "earnings_release", "overlapping"
    if diag["tables"] / max(diag["pages"], 1) > 1.0:
        return "financial_report", "overlapping"
    return "unknown", "overlapping"


def classify_htm(path: Path) -> tuple[str, str]:
    """Classify an EDGAR HTM file from its filename — no text scan needed.

    Filenames follow TICKER_FORMTYPE_DATE.htm, e.g. AMZN_10-K_2025-02-07.htm.
    """
    parts = path.stem.split("_")
    if len(parts) >= 2:
        form_type = parts[1].upper()
        if form_type == "10-K":
            return "10-K", "overlapping"
        if form_type == "10-Q":
            return "10-Q", "overlapping"
        if form_type == "8-K":
            return "8-K", "overlapping"
    return "unknown", "overlapping"


# ── Verify PDF classification ─────────────────────────────────────────────────
print(f"{'File':<45} {'Doc Type':<20} {'Mode'}")
print("-" * 80)
for diag in diagnostics:
    path = PDF_DIR / diag["file"]
    doc_type, mode = classify(path, diag)
    print(f"{diag['file']:<45} {doc_type:<20} {mode}")

File                                          Doc Type             Mode
--------------------------------------------------------------------------------
AMD Q4'25 Earnings Slides FINAL.pdf           earnings_slides      per_page
Amazon-2025-Annual-Report.pdf                 unknown              overlapping
Meta-12-31-2024-10K-ARS.pdf                   10-K                 overlapping
Meta-12-31-2025-Exhibit-99-1-FINAL.pdf        unknown              overlapping
NVIDIA-2025-Annual-Report.pdf                 10-K                 overlapping
TSLA-Q1-2026-Update.pdf                       earnings_slides      per_page
goog-10-q-q1-2025.pdf                         10-Q                 overlapping


## 4. Table Extraction

Tables are extracted as **separate chunks** from prose so that financial values keep their row/column context when embedded.

`table_to_chunk` converts any **2-D list of cell strings** to a markdown table chunk — it's the shared serialiser for both sources:
- **PDFs**: pdfplumber's `table_obj.extract()` produces the 2-D list directly
- **HTM files**: `extract_htm_table` walks the BeautifulSoup `<table>` element and builds the same 2-D list, then hands it to `table_to_chunk`

EDGAR HTML filings use deeply nested `<table>` elements for layout (indentation, two-column text). `extract_htm_table` uses `recursive=False` on its `find_all` calls so it only reads cells that are direct children of each row, not cells from nested child tables.

In [16]:
def table_to_chunk(table_data: list[list], page_num, source: str, idx: int, doc_type: str) -> Chunk | None:
    """Serialize a 2-D table (from pdfplumber or BeautifulSoup) to a markdown Chunk."""
    rows = [[str(cell or "").strip() for cell in row] for row in table_data]
    rows = [row for row in rows if any(cell for cell in row)]

    if not rows:
        return None

    header = rows[0]
    body   = rows[1:]

    header_line = "| " + " | ".join(header) + " |"
    separator   = "| " + " | ".join("---" for _ in header) + " |"
    data_lines  = []
    for row in body:
        padded = row + [""] * (len(header) - len(row))
        data_lines.append("| " + " | ".join(padded[:len(header)]) + " |")

    md = "\n".join([header_line, separator] + data_lines)

    return Chunk(
        id=make_chunk_id(source, idx, md),
        text=md,
        source=source,
        page=page_num,
        content_type="table",
        document_type=doc_type,
    )


def extract_htm_table(table_elem) -> list[list[str]]:
    """Convert a BeautifulSoup <table> to the 2-D list that table_to_chunk expects.

    recursive=False on find_all keeps us from pulling cells out of nested child
    tables — those are layout artifacts in EDGAR HTML, not data rows.
    """
    rows = []
    for tr in table_elem.find_all("tr", recursive=False):
        cells = [
            cell.get_text(separator=" ", strip=True)
            for cell in tr.find_all(["th", "td"], recursive=False)
        ]
        if any(cells):
            rows.append(cells)
    return rows


# ── Quick visual test (PDF table) ────────────────────────────────────────────
test_path = PDF_DIR / "goog-10-q-q1-2025.pdf"
with pdfplumber.open(test_path) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        tables = page.find_tables()
        if tables:
            sample = table_to_chunk(tables[0].extract(), page_num, test_path.name, 0, "10-Q")
            if sample:
                print(f"Page {page_num} — first table as markdown:")
                print(sample.text[:600])
                break

Page 1 — first table as markdown:
| he registrant is a shell company (as defined in Rule 12b-2 of the Exchange Act). Yes | ☐ | No |
| --- | --- | --- |


## 5. Prose Chunking (Overlapping Windows)

Dense reports (10-K, 10-Q) are split into **overlapping windows** of text.

Why overlapping?
- A sentence that falls at the boundary between two chunks would be cut in half
- The OVERLAP window repeats the tail of chunk N at the start of chunk N+1
- This ensures every complete sentence is fully represented in at least one chunk

```
chunk 1: |=========================|----|
chunk 2:                      |----|=========================|
                               ^^^^  overlap region
```

Table regions are **masked out** before prose extraction — otherwise text inside table cells would appear in both the table chunk and the prose chunk.

In [17]:
def extract_prose_text(page) -> str:
    """
    Extracts text from a pdfplumber page, excluding any areas covered by tables.

    pdfplumber works at the individual text-object level. Its filter() method
    accepts a predicate that receives each character/word object and returns
    True to keep it or False to discard it.

    We discard any object whose bounding box falls inside a detected table region.
    """
    # Get the bounding boxes of all tables on this page.
    # Each bbox is (x0, top, x1, bottom) in PDF coordinate space.
    table_bboxes = [t.bbox for t in page.find_tables()]

    if not table_bboxes:
        # No tables on this page — extract everything directly.
        return page.extract_text() or ""

    # Build a filtered view of the page that excludes table regions.
    # The lambda checks whether each text object's bounding box overlaps
    # with any of the table regions — if it does, the object is dropped.
    def not_in_table(obj):
        for (x0, top, x1, bottom) in table_bboxes:
            # An object is inside the table if its coordinates fall within the bbox.
            # We use a small tolerance (1pt) to avoid floating-point edge cases
            # where a text object sits exactly on the table border.
            if (x0 - 1 <= obj["x0"] and obj["x1"] <= x1 + 1 and
                top - 1 <= obj["top"] and obj["bottom"] <= bottom + 1):
                return False  # inside a table — exclude it
        return True  # outside all tables — keep it

    masked_page = page.filter(not_in_table)
    return masked_page.extract_text() or ""


def prose_to_chunks(text: str, page_num: int, source: str, start_idx: int, doc_type: str) -> list[Chunk]:
    """
    Splits a block of prose text into overlapping fixed-size chunks.

    The sliding window advances by (CHUNK_SIZE - OVERLAP) characters each step.
    This means the last OVERLAP characters of chunk N reappear at the start of
    chunk N+1, preventing sentences from being silently cut at chunk boundaries.
    """
    text = text.strip()

    # Skip near-empty text — footers, page numbers, whitespace artifacts.
    if len(text) < MIN_CHARS:
        return []

    chunks = []
    idx = start_idx
    pos = 0  # current position in the text string

    while pos < len(text):
        # Slice out a window of CHUNK_SIZE characters starting at pos.
        snippet = text[pos : pos + CHUNK_SIZE]

        if len(snippet) >= MIN_CHARS:
            chunks.append(Chunk(
                id=make_chunk_id(source, idx, snippet),
                text=snippet,
                source=source,
                page=page_num,
                content_type="prose",
                document_type=doc_type,
            ))
            idx += 1

        # Advance by CHUNK_SIZE minus OVERLAP so the next window overlaps
        # with the tail of the current one.
        pos += CHUNK_SIZE - OVERLAP

    return chunks


print("Prose chunking functions defined.")

Prose chunking functions defined.


## 6. Per-Page Chunking (Slides)

For slide-format documents (earnings presentations, investor decks), each page is one semantic unit. Merging text across pages would mix unrelated topics — slide 5 might be "Revenue" and slide 6 might be "Guidance", and concatenating them produces noise.

Strategy: **one Chunk per page**, containing all text extracted from that page. Tables on slide pages are small and tightly bound to their surrounding context, so we don't separate them — the whole page goes into a single chunk.

In [18]:
def page_to_chunk(page, page_num: int, source: str, idx: int, doc_type: str) -> Chunk | None:
    """
    Extracts all text from a single slide page as one Chunk.

    Unlike the overlapping prose strategy, we do NOT mask table regions here.
    Slide tables are small (usually 3-6 cells) and provide essential context
    for the surrounding bullet points — keeping them together improves retrieval.
    """
    # extract_text() returns all text objects on the page in reading order.
    # For slides, this typically gives us the title + bullet points as one block.
    text = (page.extract_text() or "").strip()

    if len(text) < MIN_CHARS:
        # Skip near-empty slides — title-only pages, divider slides, etc.
        return None

    return Chunk(
        id=make_chunk_id(source, idx, text),
        text=text,
        source=source,
        page=page_num,
        content_type="prose",
        document_type=doc_type,
    )


print("Per-page chunking function defined.")

Per-page chunking function defined.


## 7. Parsers

`parse_pdf` and `parse_htm` both return a flat `list[Chunk]` — everything above this cell is shared between them.

`parse()` is the single entry point: it dispatches on file extension so the pipeline cell doesn't need to know what format it's processing.

In [19]:
def parse_pdf(path: Path) -> list[Chunk]:
    """diagnose → classify → chunk every page of a PDF."""
    source = path.name
    diag = diagnose(path)
    doc_type, mode = classify(path, diag)

    if diag["likely_scan"]:
        print(f"  [SKIP] {source} — scanned document (OCR not implemented)")
        return []

    chunks = []
    idx = 0

    with pdfplumber.open(path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            if mode == "per_page":
                chunk = page_to_chunk(page, page_num, source, idx, doc_type)
                if chunk:
                    chunks.append(chunk)
                    idx += 1
            else:
                for table_obj in page.find_tables():
                    chunk = table_to_chunk(table_obj.extract(), page_num, source, idx, doc_type)
                    if chunk:
                        chunks.append(chunk)
                        idx += 1
                prose_text = extract_prose_text(page)
                new_prose = prose_to_chunks(prose_text, page_num, source, idx, doc_type)
                chunks.extend(new_prose)
                idx += len(new_prose)

    return chunks


def parse_htm(path: Path) -> list[Chunk]:
    """classify (from filename) → extract tables → extract prose from an EDGAR HTM file."""
    source = path.name
    doc_type, _ = classify_htm(path)

    parts = path.stem.split("_")
    extra = {}
    if len(parts) >= 1:
        extra["ticker"] = parts[0]
    if len(parts) >= 3:
        extra["filing_date"] = parts[2]

    soup = BeautifulSoup(
        path.read_text(encoding="utf-8", errors="replace"),
        "html.parser",
    )
    for tag in soup(["script", "style", "head"]):
        tag.decompose()

    chunks = []
    idx = 0

    # Only top-level tables — nested ones are layout artifacts in EDGAR HTML.
    # Collect before decompose() so the tree walk isn't invalidated mid-loop.
    top_tables = [t for t in soup.find_all("table") if not t.find_parent("table")]
    for table_elem in top_tables:
        chunk = table_to_chunk(extract_htm_table(table_elem), None, source, idx, doc_type)
        if chunk:
            chunk.extra.update(extra)
            chunks.append(chunk)
            idx += 1
        table_elem.decompose()

    lines = [l for l in soup.get_text(separator="\n", strip=True).splitlines() if l.strip()]
    prose_chunks = prose_to_chunks("\n".join(lines), None, source, idx, doc_type)
    for c in prose_chunks:
        c.extra.update(extra)
    chunks.extend(prose_chunks)

    return chunks


def parse(path: Path) -> list[Chunk]:
    """Dispatch to parse_pdf or parse_htm based on file extension."""
    ext = path.suffix.lower()
    if ext == ".pdf":
        return parse_pdf(path)
    if ext in (".htm", ".html"):
        return parse_htm(path)
    print(f"  [SKIP] {path.name} — unsupported format '{ext}'")
    return []


print("parse_pdf, parse_htm, and parse defined.")

parse_pdf, parse_htm, and parse defined.


## 8. Run the Pipeline

Parse every `.pdf` and `.htm` file in `datasets/client/` through the unified `parse()` entry point.

In [ ]:
all_chunks: list[Chunk] = []

for doc_path in sorted(PDF_DIR.iterdir()):
    if doc_path.suffix.lower() not in (".pdf", ".htm", ".html"):
        continue
    chunks = parse(doc_path)
    all_chunks.extend(chunks)
    n_tables = sum(1 for c in chunks if c.content_type == "table")
    n_prose  = sum(1 for c in chunks if c.content_type == "prose")
    print(f"{doc_path.name:<50}  {len(chunks):>4} chunks  ({n_tables} tables, {n_prose} prose)")

print(f"\nTotal chunks across all documents: {len(all_chunks)}")

## 9. Inspect the Output

Spot-check chunks before moving on to embedding.

In [ ]:
import pandas as pd

rows = [
    {
        "source":        c.source,
        "document_type": c.document_type,
        "content_type":  c.content_type,
        "page":          c.page,
        "chars":         len(c.text),
    }
    for c in all_chunks
]
df = pd.DataFrame(rows)

print("=== Chunk counts by document type and content type ===")
print(df.groupby(["document_type", "content_type"]).size().unstack(fill_value=0))
print("\n=== Chunk size distribution (characters) ===")
print(df["chars"].describe().round(1))

In [ ]:
table_chunks = [c for c in all_chunks if c.content_type == "table"]
if table_chunks:
    sample = table_chunks[0]
    print(f"Source : {sample.source}  (page {sample.page})")
    print(f"Type   : {sample.document_type}")
    print()
    print(sample.text[:800])

## 10. Check for ID Collisions

Every chunk needs a unique ID before it can be stored in ChromaDB — duplicate IDs cause silent overwrites.

In [ ]:
all_ids = [c.id for c in all_chunks]
unique_ids = set(all_ids)
n_duplicate = len(all_ids) - len(unique_ids)

print(f"Total chunks : {len(all_ids)}")
print(f"Unique IDs   : {len(unique_ids)}")
print(f"Duplicates   : {n_duplicate}")

if n_duplicate == 0:
    print("\nAll IDs are unique — safe to ingest into ChromaDB.")
else:
    from collections import Counter
    dupes = [id_ for id_, count in Counter(all_ids).items() if count > 1]
    print(f"\nDuplicate IDs found: {dupes[:10]}")